<a href="https://colab.research.google.com/github/supriya-006/FlyRank_Assignment/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/supriya-006/FlyRank_Assignment/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [9]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

In [10]:
# Confirm dtypes for the two flag columns before trusting boolean logic on them
con.sql(f"""
DESCRIBE SELECT client_has_gsc, client_has_ga4
FROM '{BASE}/fact_content_daily_performance/month={MONTH}/data_0.parquet' LIMIT 0
""").show()

features = con.sql(f"""
SELECT
    content_hash_id,
    report_date,
    AVG(sessions_paid) OVER w   AS sessions_paid_7d_avg,
    AVG(sessions_direct) OVER w AS sessions_direct_7d_avg,
    SUM(ai_gemini + ai_claude + ai_meta) OVER w AS ai_referral_7d_total,
    AVG(scroll_events) OVER w  AS scroll_events_7d_avg,
    (client_has_gsc AND client_has_ga4) AS has_full_tracking
FROM '{BASE}/fact_content_daily_performance/month={MONTH}/data_0.parquet'
WINDOW w AS (
    PARTITION BY content_hash_id
    ORDER BY report_date
    RANGE BETWEEN INTERVAL 6 DAYS PRECEDING AND CURRENT ROW
)
""").df()

features.head()

┌────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type │  null   │   key   │ default │  extra  │
│    varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_has_gsc │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4 │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,report_date,sessions_paid_7d_avg,sessions_direct_7d_avg,ai_referral_7d_total,scroll_events_7d_avg,has_full_tracking
0,content_b8f7f3e624555e36,2026-03-05,NaN,NaN,NaN,NaN,False
1,content_b8f7f3e624555e36,2026-03-06,NaN,NaN,NaN,NaN,False
2,content_b8f7f3e624555e36,2026-03-07,NaN,NaN,NaN,NaN,False
3,content_b8f7f3e624555e36,2026-03-08,NaN,NaN,NaN,NaN,False
4,content_b8f7f3e624555e36,2026-03-09,NaN,NaN,NaN,NaN,False


**Five features, built from confirmed columns:**

1. `sessions_paid_7d_avg` — trailing 7-day average of `sessions_paid` per content item.
   Knowable at the decision moment because it only uses `report_date` values on or
   before that date.
2. `sessions_direct_7d_avg` — trailing 7-day average of `sessions_direct`, same window logic.
   Knowable at the decision moment because it's strictly backward-looking.
3. `ai_referral_7d_total` — trailing 7-day sum of `ai_gemini + ai_claude + ai_meta`.
   Knowable at the decision moment because AI-referral traffic on past days is
   already logged by the time a decision is made.
4. `scroll_events_7d_avg` — trailing 7-day average of `scroll_events`.
   Knowable at the decision moment because engagement events are recorded same-day
   or earlier, never in advance.
5. `has_full_tracking` — `client_has_gsc AND client_has_ga4` (static per client).
   Knowable at the decision moment because it's a client setup fact, not an outcome —
   true or false regardless of which day you ask.

Dtypes for `client_has_gsc`/`client_has_ga4` aren't confirmed yet (could be BOOLEAN or
0/1 INTEGER) — the code below checks both.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Feature notes:**

1. **`sessions_paid_7d_avg`** — trailing 7-day average of paid-channel sessions
   for a content item. Missing/NULL when a content item has fewer than 7 days
   of history in the panel (early rows) or when `sessions_paid` itself is NULL
   for every day in the window — left as NULL rather than filled with 0, since
   0 and "no data yet" mean different things here. Numeric, not categorical.
   **Available before the decision moment**: yes — only ever averages
   `report_date` values on or before the row's own date.

2. **`sessions_direct_7d_avg`** — same construction as above, on `sessions_direct`.
   Same missing-value handling (NULL stays NULL, no fill). Numeric.
   **Available before the decision moment**: yes, same backward-only window.

3. **`ai_referral_7d_total`** — trailing 7-day sum of `ai_gemini + ai_claude + ai_meta`
   referral counts. If any of the three underlying columns is NULL on a given day,
   the summed row is NULL for that day (not treated as 0) — this will need a
   decision later (impute vs. drop) once I check how often it actually happens.
   Numeric. **Available before the decision moment**: yes.

4. **`scroll_events_7d_avg`** — trailing 7-day average of scroll engagement events.
   Same NULL-stays-NULL handling as the session averages. Numeric.
   **Available before the decision moment**: yes.

5. **`has_full_tracking`** — boolean/derived flag: `client_has_gsc AND client_has_ga4`.
   Not really "missing" in the usual sense — it's a per-client setup fact, so it's
   either true, false, or (if one of the source flags is itself NULL) unknown for
   that client; treated as categorical (2–3 levels), not numeric.
   **Available before the decision moment**: yes — it describes client setup, not
   an outcome, so it's true on day 1 as much as on any later day.

**General handling note**: none of these five get filled with 0 or a mean — a
missing trailing average means "not enough history yet," which is a real state
worth keeping distinct from "value was 0." Any imputation decision gets made
explicitly later, not silently here.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**The leakage hunt:**

I'm testing the same five features from section 1 against a label I haven't
used yet: `session_growth_flag` — whether a content item's total sessions
(`sessions_paid + sessions_direct`) over the *next* 7 days exceed the same sum
over the *prior* 7 days. That's a genuinely future-looking label, which makes
it a fair target for the trap: add one column built from that same future
window as a "feature," watch the score jump, then remove it.

The one deliberately-leaked column: `future_sessions_total_7d` — literally the
forward-window sum the label is derived from. No model should legitimately see
this at decision time; including it should make the score jump toward perfect,
which is the whole point of showing it and then killing it.

In [ ]:
# Build label + honest features + one deliberate leak, all from real columns
panel = con.sql(f"""
SELECT
    content_hash_id,
    report_date,
    AVG(sessions_paid) OVER w7   AS sessions_paid_7d_avg,
    AVG(sessions_direct) OVER w7 AS sessions_direct_7d_avg,
    SUM(ai_gemini + ai_claude + ai_meta) OVER w7 AS ai_referral_7d_total,
    AVG(scroll_events) OVER w7   AS scroll_events_7d_avg,
    (client_has_gsc AND client_has_ga4) AS has_full_tracking,
    SUM(sessions_paid + sessions_direct) OVER w7   AS past_sessions_total_7d,
    SUM(sessions_paid + sessions_direct) OVER wfut AS future_sessions_total_7d
FROM '{BASE}/fact_content_daily_performance/month={MONTH}/data_0.parquet'
WINDOW
    w7   AS (PARTITION BY content_hash_id ORDER BY report_date
              RANGE BETWEEN INTERVAL 6 DAYS PRECEDING AND CURRENT ROW),
    wfut AS (PARTITION BY content_hash_id ORDER BY report_date
              RANGE BETWEEN CURRENT ROW AND INTERVAL 6 DAYS FOLLOWING)
""").df()

panel["session_growth_flag"] = (
    panel["future_sessions_total_7d"] > panel["past_sessions_total_7d"]
).astype(int)
panel = panel.dropna()

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = [
    "sessions_paid_7d_avg", "sessions_direct_7d_avg",
    "ai_referral_7d_total", "scroll_events_7d_avg", "has_full_tracking",
]
leak_features = honest_features + ["future_sessions_total_7d"]

def quick_score(feature_cols):
    X = panel[feature_cols].astype(float)
    y = panel["session_growth_flag"]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
    model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    return roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])

honest_auc = quick_score(honest_features)
leaked_auc = quick_score(leak_features)

print(f"Honest AUC (no leak):     {honest_auc:.3f}")
print(f"Leaked AUC (with trap):   {leaked_auc:.3f}")

# Delete the leak, keep only the honest number going forward
del panel["future_sessions_total_7d"]
print(f"\nFinal honest score kept: {honest_auc:.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



**Excluded fields, with why:**

- **`future_sessions_total_7d`** — the column from the leakage hunt in section 3.
  Built directly from the same forward window the label is derived from; keeping
  it would mean the model is scored on seeing its own answer.

- **`session_growth_flag`** (the label itself) — excluded from the feature set
  by definition. A label is never also a feature; including it isn't leakage in
  the technical sense, it's just wrong.

- **`fact_content_query_90d` (the whole table)** — its 90-day window with
  last-30/prev-30 sub-windows can cross into dates after any given decision
  point for a March row. Until I've explicitly checked which dates in that
  window fall before vs. after each decision date, I'm not joining it in.

- **`client_hash_id` / `content_hash_id`** — used as join/partition keys, never
  as model features. They identify a row, they don't carry predictive signal,
  and treating a hash as a numeric feature would just be noise.

- **`report_date`** raw value — used to build the trailing/forward windows, not
  fed to the model directly. A raw calendar date isn't a feature on its own;
  the trailing aggregates built from it are.

- **Anything from `dim_clients`** (e.g. `gsc_data_start`, `ga4_data_start`) —
  not touched here. These describe *when a client started being tracked*, which
  correlates with unbalanced-panel effects (named as this lane's limitation in
  the contract notebook) rather than with content performance itself. Mixing
  them in without a specific reason would blur tenure effects into the model
  silently.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.